### Middleware
Midleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tools selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detecction.


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Sumamarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history
* Applications where preserving full conversation context matters.

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Messagebased summarization
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ]
) 


In [19]:
# Run with thread id
config = {"configurable":{"thread_id":"test_1"}}

In [ ]:
# the same output with more detail
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for i,q in enumerate (questions, start=1):
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"\n--- Iteration {i} ---")
    print(f"Question: {q}")
    #print(f"Messages: {response}")
    print(f"total messages:{len(response['messages'])}")
    for message in response["messages"]:
        source = message.additional_kwargs.get("lc_source")
        if source == "summarization":
            print(f"[SUMMARY]: {message.content}")
        else:
            print(
                f"[{message.type.upper()}]: "
                f"{message.content}"
                )
    


--- Iteration 1 ---
Question: What is 2+2?
total messages:2
[HUMAN]: What is 2+2?
[AI]: 2 + 2 equals 4.

--- Iteration 2 ---
Question: What is 10*5?
total messages:4
[HUMAN]: What is 2+2?
[AI]: 2 + 2 equals 4.
[HUMAN]: What is 10*5?
[AI]: 10 * 5 equals 50.

--- Iteration 3 ---
Question: What is 100/4?
total messages:6
[HUMAN]: What is 2+2?
[AI]: 2 + 2 equals 4.
[HUMAN]: What is 10*5?
[AI]: 10 * 5 equals 50.
[HUMAN]: What is 100/4?
[AI]: 100 / 4 equals 25.

--- Iteration 4 ---
Question: What is 15-7?
total messages:8
[HUMAN]: What is 2+2?
[AI]: 2 + 2 equals 4.
[HUMAN]: What is 10*5?
[AI]: 10 * 5 equals 50.
[HUMAN]: What is 100/4?
[AI]: 100 / 4 equals 25.
[HUMAN]: What is 15-7?
[AI]: 15 - 7 equals 8.

--- Iteration 5 ---
Question: What is 3*3?
total messages:10
[HUMAN]: What is 2+2?
[AI]: 2 + 2 equals 4.
[HUMAN]: What is 10*5?
[AI]: 10 * 5 equals 50.
[HUMAN]: What is 100/4?
[AI]: 100 / 4 equals 25.
[HUMAN]: What is 15-7?
[AI]: 15 - 7 equals 8.
[HUMAN]: What is 3*3?
[AI]: 3 * 3 equals 9.

### Token Size

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool


@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn -4 start, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi """

# Messagebased summarization
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
) 

config = {"configurable":{"thread_id":"test_1"}}

#Token  counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [ ]:
# Run test
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]
for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")


Paris: ~144 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='ed1040a7-d0a1-48d5-8f73-a1d04a733d8b'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4a812fa9ae', 'id': 'chatcmpl-EfrUses6nJCI2o3vxU0o259YkaqEi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19d9e-e27d-75b1-9e56-f78646ee7539-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_MiZislLpT7RNvR6

### Fraction

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool


@tool
def search_hotels(city:str)->str:
    """Search hotels."""
    return f"Hotels in {city}:Grand Hotel $350, City Inn $180, Budget Stay $75"

# Low Fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005), # 0.5% = ~640 tokens
            keep=("fraction", 0.002), # 0.2% = ~256 tokens
        ),
    ]
) 

config = {"configurable":{"thread_id":"test_1"}}

#Token  counter
def count_tokens(messages):
    return  sum(len(str(m.content)) for m in messages) // 4

#Test
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]
for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~66 tokens (0.0516%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='102f7929-ab40-44b3-8ef6-fe59d4eeff17'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 44, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_16c1774bc0', 'id': 'chatcmpl-Efrj7NljMgkKJEdGXtDPpkRngnDuQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19dac-5bc3-71ef-8524-2ef2107e4789-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_m5PsqcCCa3c6hhw

### Human in the Loop 
Pause Agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
* High-stakes operations requiring human approval (e.g.database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock function to send an email."""
    return f"Email sent to :{recipient} with subject '{subject}'"


In [11]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = "gpt-4o",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [14]:
config = {"configurable":{"thread_id":"test-approve"}}
# step 1: Request
result = agent.invoke(
    {
        "messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'how are you?'")]
    },
    config = config
)

In [15]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello how are you?'", additional_kwargs={}, response_metadata={}, id='578b6993-6a6e-4f15-9cff-edb233362634'),
  AIMessage(content='Could you please provide the body of the email that you would like to send to john@test.com?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 93, 'total_tokens': 114, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c2c459cd0', 'id': 'chatcmpl-EftygsCHt7TkC1U80BSacURWHuI89', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19e30-5a45-777b-b97e-53

In [17]:
from langgraph.types import Command
# step 2: Approve
if "__interrupt__" in result:
    print("Paused! Approving...")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result:{result['messages'][-1].content}")

Paused! Approving...
Result:The email has been sent to john@test.com with the subject "Hello" and the body "how are you?".


In [18]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello how are you?'", additional_kwargs={}, response_metadata={}, id='578b6993-6a6e-4f15-9cff-edb233362634'),
  AIMessage(content='Could you please provide the body of the email that you would like to send to john@test.com?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 93, 'total_tokens': 114, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c2c459cd0', 'id': 'chatcmpl-EftygsCHt7TkC1U80BSacURWHuI89', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a19e30-5a45-777b-b97e-53

### Reject

In [19]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = "gpt-4o",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [20]:
config = {"configurable":{"thread_id":"test-reject"}}
# step 1: Request
result = agent.invoke(
    {
        "messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'how are you?'")]
    },
    config = config
)

In [ ]:
from langgraph.types import Command
# step 2: Reject
if "__interrupt__" in result:
    print("Paused! Rejecting...")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result:{result['messages'][-1].content}")

Paused! Approving...
Result:It seems like the email wasn't sent. If you need help with anything else or want to try sending it again, please let me know!


In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='42a3f7f5-5f63-481c-8791-f97a36cca6a7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-EfuYjJ8dJFxguwBrCuOn8d2otezXx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19e52-79e7-7068-af48-5657e0e601d9-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient'

### Edit

In [50]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock function to send an email."""
    return f"Email sent to :{recipient} with subject '{subject}'"

agent = create_agent(
    model = "gpt-4o",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [51]:
config = {"configurable":{"thread_id":"test-edit"}}
# step 1: Request (with wrong info)
result = agent.invoke(
    {
        "messages":[HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'how are you?'")]
    },
    config = config
)

In [53]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='81760b99-b22d-4bf0-817c-e2333ddf5eb6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c2c459cd0', 'id': 'chatcmpl-EfvRdZgYTqPlzPAySeR5jtlQYLdTn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19e86-6dac-77ac-9907-93a57a730804-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient

In [54]:
from langgraph.types import Command
# step 2: Approve
if "__interrupt__" in result:
    print("Paused! Editing...")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool", #tool name
                            "args":{ # new arguments
                                "recipient":"correct@email.com",
                                "subject":"corrected subject",
                                "body":"This was edited by human before sending"
                            }
                        }
                   }
                ]
            }
        ),
        config=config
    )
    print(f"Result:{result['messages'][-1].content}")

Paused! Editing...
Result:I've sent the email, but it was edited by a human reviewer before sending. If you need further assistance or corrections, feel free to let me know!


In [56]:
from langchain_core.messages import (
    AIMessage,
    ToolMessage
)

for message in result["messages"]:

    if isinstance(message, AIMessage) and message.tool_calls:
        print("\nOriginal tool call:")
        print(message.tool_calls)

    elif isinstance(message, ToolMessage):
        print("\nExecuted tool result:")
        print(message.content)

    elif isinstance(message, AIMessage):
        print("\nFinal AI response:")
        print(message.content)


Original tool call:
[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'subject': 'Test', 'body': 'how are you?'}, 'id': 'call_AmVJnjFAHQI1IJSTXWipckLQ', 'type': 'tool_call'}]

Executed tool result:
Note: a human reviewer replaced this tool call before it ran. The call recorded in your message is the one you produced, not the one that executed. This was intentional and authorized. Do not re-issue your original call. Executed instead: send_email_tool with arguments {"recipient": "correct@email.com", "subject": "corrected subject", "body": "This was edited by human before sending"}.

Tool response:
Email sent to :correct@email.com with subject 'corrected subject'

Final AI response:
I've sent the email, but it was edited by a human reviewer before sending. If you need further assistance or corrections, feel free to let me know!
